# 01. 탐색적 데이터 분석 (EDA)

> README **5.1 데이터 개요**, **5.2 데이터 특징**, **6.1 탐색적 데이터 분석(발견 1~4)**, **6.2 전처리 결정의 근거**에 대응하는 노트북입니다.

**목적**
모델링 전에 데이터 구조, Target 불균형 정도, 데이터 품질, Feature 분포와 Feature–Target 관계, 결함들 사이의 구조를 확인하고, 이를 근거로 전처리(02)와 모델링(03) 방향을 정합니다.

**원칙**
- 전처리·모델링 방향을 정하는 EDA(1~7장, 11장)는 **Train 데이터(1,552건)만** 사용합니다. Test 정보가 분석 판단에 섞이지 않도록 하기 위함입니다.
- 데이터 구조가 평가 점수에 주는 영향을 보는 분석(8~10장의 일부)은 전체 데이터 또는 Test 결과를 함께 씁니다. 해당 셀마다 **[전체 데이터]**, **[Test 포함]**으로 표시했고, 이 결과는 모델이나 설정을 고르는 데 쓰지 않습니다.
- 전처리 결정의 근거를 찾는 단계이므로 **원본 데이터(`data/raw/Faults.csv`)**를 사용합니다. 실제 전처리는 02에서 수행합니다.
- 분할 인덱스는 `src/make_split.py`로 만든 팀 공통 `data/split/train_idx.csv`를 사용합니다.

| 섹션 | 내용 | README |
|---|---|---|
| 1 | 데이터 불러오기 및 기본 구조 | 5.1 |
| 2 | 변수 묶음과 직접 확인한 사실 | 5.2 |
| 3 | Target 분포와 불균형 정도 | 5.1, 5.2 |
| 4 | 데이터 품질 (결측·중복·값 범위·Target 구조·강종 변수) | 6.2 |
| 5 | Feature 분포 (왜도·극단값) | 6.2 |
| 6 | Feature 간 관계 (상관) | 6.2 |
| 7 | 발견 1. 변수 하나의 분리력 → `figures/eda_result.png` | 6.1 |
| 8 | 발견 2. 생산 조건 묶음 → `figures/process_condition.png` | 6.1 |
| 9 | 발견 3. 모양만으로 구분되는 정도 → `figures/shape_similarity.png` | 6.1 |
| 10 | 발견 4. 같은 자리에서 반복되는 결함 → `figures/roll_mark_period.png` | 6.1 |
| 11 | Other_Faults 내부 특성 | 8.2 |
| 12 | EDA 결론 및 전처리·모델링 결정 | 6.2 |

In [ ]:
import sys
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 노트북 위치와 무관하게 프로젝트 루트(src/config.py가 있는 폴더)를 찾는다
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

# 한글 폰트: OS별로 있는 폰트를 순서대로 사용
plt.rcParams["font.family"] = ["Malgun Gothic", "AppleGothic", "NanumGothic",
                               "Noto Sans CJK KR", "Noto Sans CJK JP", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
pd.set_option("display.width", 200)

## 1. 데이터 불러오기 및 기본 구조 (README 5.1)

원본 데이터에서 Train 인덱스에 해당하는 행만 사용합니다. Target은 7개 원-핫 열 중 1인 열의 이름으로 만들고, 표기 통일을 위해 라벨명 `K_Scatch`만 `K_Scratch`로 바꿔 표시합니다.

In [ ]:
from config import TARGET_COLS   # 원본 열 이름 기준 (K_Scatch 포함)

raw = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "Faults.csv")
train_idx = pd.read_csv(PROJECT_ROOT / "data" / "split" / "train_idx.csv")["index"]

X_train = raw.drop(columns=TARGET_COLS).loc[train_idx]
y_train = (raw[TARGET_COLS].idxmax(axis=1)
           .replace({"K_Scatch": "K_Scratch"})
           .loc[train_idx])

print("원본 전체:", raw.shape)
print("X_train  :", X_train.shape)
print("y_train  :", y_train.shape)
X_train.head()

In [ ]:
print("[데이터 타입]")
print(X_train.dtypes.value_counts().to_string())
X_train.describe().T

**확인 결과**
- Train은 1,552건, 입력 Feature 27개이며 모두 수치형(int 14, float 13)입니다. 범주형 인코딩은 필요하지 않습니다.
- `Y_Minimum`(최대 약 1,300만)과 `Edges_Index`(0~1)처럼 Feature 간 값의 범위 차이가 매우 큽니다. → Scale에 민감한 모델은 Scaling이 필요합니다.
- 면적·둘레·밝기 합계는 평균이 중앙값보다 훨씬 커서 치우친 분포가 예상됩니다. → 5장에서 확인합니다.

## 2. 변수 묶음과 직접 확인한 사실 (README 5.2)

데이터 제공처의 공식 변수 설명서가 없으므로, 변수를 의미에 따라 다섯 묶음으로 나누고 이름만으로는 알 수 없는 성질을 데이터로 직접 확인합니다. 이 묶음은 이후 그림의 색 구분에도 씁니다.

In [ ]:
VAR_GROUP = {
    "위치": ["X_Minimum", "X_Maximum", "Y_Minimum", "Y_Maximum"],
    "크기": ["Pixels_Areas", "X_Perimeter", "Y_Perimeter", "LogOfAreas", "Log_X_Index",
            "Log_Y_Index", "SigmoidOfAreas", "Sum_of_Luminosity"],
    "모양": ["Edges_Index", "Empty_Index", "Square_Index", "Outside_X_Index", "Edges_X_Index",
            "Edges_Y_Index", "Outside_Global_Index", "Orientation_Index"],
    "밝기": ["Minimum_of_Luminosity", "Maximum_of_Luminosity", "Luminosity_Index"],
    "공정 조건": ["TypeOfSteel_A300", "TypeOfSteel_A400", "Steel_Plate_Thickness", "Length_of_Conveyer"],
}
GROUP_COLOR = {"크기": "#2E6FB7", "모양": "#6BAED6", "밝기": "#9E9AC8", "위치": "#9AA3AD", "공정 조건": "#C9443C"}
feat2group = {f: g for g, fs in VAR_GROUP.items() for f in fs}
assert set(feat2group) == set(X_train.columns), "묶음에 빠지거나 남는 변수가 있습니다."
pd.Series(feat2group).groupby(lambda f: feat2group[f]).size().to_frame("변수 수")

In [ ]:
# ① Length_of_Conveyer는 컨베이어 길이가 아니라 강판(검사 영역)의 가로 폭인가?
print("결함 오른쪽 끝(X_Maximum) ≤ Length_of_Conveyer 인 행 비율:",
      (X_train["X_Maximum"] <= X_train["Length_of_Conveyer"]).mean())

# ② Sum_of_Luminosity는 밝기 변수인가, 크기 변수인가?
print("Pixels_Areas ↔ Sum_of_Luminosity Spearman:",
      round(X_train["Pixels_Areas"].corr(X_train["Sum_of_Luminosity"], method="spearman"), 4))

# ③ 비율·모양 지표가 원래 측정값으로 계산된 값인가? (다시 계산해서 비교)
dx = X_train["X_Maximum"] - X_train["X_Minimum"]
dy = X_train["Y_Maximum"] - X_train["Y_Minimum"]
L = X_train["Length_of_Conveyer"]
recomputed = {
    "LogOfAreas": np.log10(X_train["Pixels_Areas"]),
    "Log_X_Index": np.log10(dx),
    "Log_Y_Index": np.log10(dy),
    "Square_Index": np.minimum(dx, dy) / np.maximum(dx, dy),
    "Empty_Index": 1 - X_train["Pixels_Areas"] / (dx * dy),
    "Outside_X_Index": dx / L,
    "Edges_X_Index": dx / X_train["X_Perimeter"],
    "Edges_Y_Index": dy / X_train["Y_Perimeter"],
    "Edges_Index": 2 * np.minimum(X_train["X_Minimum"], L - X_train["X_Maximum"]) / L,
    "Orientation_Index": np.where(dy >= dx, 1 - dx / dy, -(1 - dy / dx)),
    "Outside_Global_Index": np.where(dy > dx, 1.0, np.where(dy == dx, 0.5, 0.0)),
    "Luminosity_Index": (X_train["Sum_of_Luminosity"] / X_train["Pixels_Areas"] - 128) / 128,
}
# SigmoidOfAreas: 결함을 둘러싼 네모 칸의 넓이(dx × dy)에 대한 로지스틱 함수. 중심·폭 상수는 공개되지 않아 데이터로 추정
from scipy.optimize import curve_fit
box = (dx * dy).astype(float)
logistic = lambda a, center, scale: 1 / (1 + np.exp(-(a - center) / scale))
(center, scale), _ = curve_fit(logistic, box, X_train["SigmoidOfAreas"], p0=[300, 150])
recomputed["SigmoidOfAreas"] = logistic(box, center, scale)
print(f"SigmoidOfAreas 추정 상수: 중심 {center:.2f}, 폭 {scale:.2f}")

check = pd.DataFrame({c: [np.abs(v - X_train[c]).max()] for c, v in recomputed.items()}, index=["최대 오차"]).T
check["일치 (오차 < 0.001)"] = check["최대 오차"] < 1e-3
print(f"\n다시 계산해서 일치한 지표: {int(check.iloc[:, 1].sum())} / {len(check)}")
check

**확인 결과**
- 모든 행에서 결함 오른쪽 끝이 `Length_of_Conveyer`를 넘지 않습니다. → 이 변수는 이름과 달리 **강판(검사 영역)의 가로 폭**입니다. 이름이 헷갈리지만 이후에도 **원본 컬럼명 `Length_of_Conveyer` 그대로** 표기합니다.
- `Sum_of_Luminosity`는 면적과 Spearman 0.98로, 이름은 밝기지만 **사실상 크기 변수**입니다. 그래서 "크기" 묶음에 넣었습니다.
- 비율·모양 지표 13개 모두 원래 측정값(좌표·면적·둘레·밝기 합·`Length_of_Conveyer`)으로 다시 계산한 값과 일치합니다. `SigmoidOfAreas`는 결함 칸의 넓이(dx × dy)에 대한 로지스틱 함수이며, 공개되지 않은 중심·폭 상수만 데이터로 추정했습니다.
- 원본에 이미 계산된 비율 지표가 들어 있으므로, 비슷한 비율을 새로 만드는 파생변수의 추가 효과는 작을 수 있습니다(README 7.5).
- `TypeOfSteel_A300`·`A400`의 관계는 4장에서 확인합니다.

## 3. Target 분포와 불균형 정도 (README 5.1, 5.2)

불균형을 "있다/없다"로만 판단하지 않고, 유형별 실제 건수와 비율을 함께 확인합니다.

In [ ]:
target_summary = pd.DataFrame({
    "Count": y_train.value_counts(),
    "Ratio(%)": (y_train.value_counts(normalize=True) * 100).round(2),
})
display(target_summary)

cnt = target_summary["Count"]
print(f"최다 유형: {cnt.idxmax()} ({cnt.max()}건)")
print(f"최소 유형: {cnt.idxmin()} ({cnt.min()}건)")
print(f"최대/최소 비율: {cnt.max() / cnt.min():.2f}배")
print(f"상위 3개 유형 비중: {target_summary['Ratio(%)'].head(3).sum():.1f}%")

In [ ]:
plot_df = target_summary.sort_values("Count")
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(plot_df.index, plot_df["Count"], color="#4C72B0")
ax.bar_label(bars, labels=[f"{c}건 ({r:.1f}%)" for c, r in
                           zip(plot_df["Count"], plot_df["Ratio(%)"])], padding=4)
ax.set_xlim(0, plot_df["Count"].max() * 1.25)
ax.set_xlabel("건수")
ax.set_title("Train 데이터 결함 유형 분포")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

**확인 결과**
- Other_Faults가 538건(34.66%)으로 가장 많고 Dirtiness가 44건(2.84%)으로 가장 적어 **최대/최소 12.23배** 차이가 있습니다.
- 상위 3개 유형(Other_Faults, Bumps, K_Scratch)이 Train의 약 75.5%를 차지합니다.

**판단**
- Accuracy는 다수 유형의 성능에 좌우되므로, **모든 유형의 F1을 같은 비중으로 평균하는 Macro F1**을 핵심 지표로 사용합니다.
- 단, "건수가 적은 유형 = 분류가 어려운 유형"이라고 미리 단정하지 않고, Baseline의 유형별 성능으로 확인합니다. 건수와 Baseline F1을 함께 본 README 5.2 그림(`figures/target_distribution.png`)은 02 노트북에서 만듭니다.
- README 5.1의 클래스 비율(34.67% ~ 2.83%, 12.24배)은 전체 데이터 기준이고, 여기 값은 Train 기준입니다. 층화 분할이라 거의 같습니다.

## 4. 데이터 품질 (README 6.2)

각 항목을 **확인 → 처리 필요 여부 판단** 순서로 봅니다.

In [ ]:
print("결측치 합계:", int(X_train.isna().sum().sum()))
print("중복 행 수 :", int(X_train.duplicated().sum()))

# Target 구조: 모든 행에서 원-핫 7개 열 중 정확히 1개만 1인가?
onehot_sum = raw.loc[train_idx, TARGET_COLS].sum(axis=1)
print("원-핫 합이 1이 아닌 행:", int((onehot_sum != 1).sum()))

In [ ]:
range_summary = pd.DataFrame({
    "min": X_train.min(),
    "max": X_train.max(),
    "n_unique": X_train.nunique(),
})
range_summary

In [ ]:
# 강종 변수 두 개가 같은 정보를 중복 표현하는지 확인
X_train[["TypeOfSteel_A300", "TypeOfSteel_A400"]].value_counts().sort_index()

**확인 결과**
- 결측치·중복 행은 0건입니다. → 처리하지 않습니다.
- 모든 행에서 원-핫 정답 열의 합이 1입니다. → 7개 열을 **단일 다중 분류 Target**으로 바꿀 수 있습니다. 이 정답 열들이 입력에 남으면 그대로 정답을 알려주는 **데이터 누수**가 되므로 반드시 제거합니다.
- 값의 범위에서 데이터 오류로 판단할 만한 값(음수 면적 등)은 보이지 않습니다.
- `TypeOfSteel_A300`과 `TypeOfSteel_A400`은 (0,1) 또는 (1,0)만 존재합니다. 두 열은 같은 정보를 반대로 표현하므로 **하나만 남겨도 정보 손실이 없습니다.**
- `Outside_Global_Index`는 고유값이 3개뿐인 이산형 지표입니다(2장에서 확인한 대로 결함 칸의 가로·세로 중 어느 쪽이 긴지).
- 측정값이 **거의 같은 행(유사 중복)**은 Train과 Test에 나뉘어 있을 수 있으므로, 전체 데이터 기준으로 02 노트북 1장에서 확인합니다.

## 5. Feature 분포 (README 6.2)

왜도가 큰 Feature를 확인하고, 극단값이 데이터 오류인지 결함 특성인지 판단합니다.

In [ ]:
skew = X_train.skew().sort_values(key=abs, ascending=False)
skew_df = skew.to_frame("skewness")
print("|왜도| ≥ 1 인 Feature")
skew_df[skew_df["skewness"].abs() >= 1]

In [ ]:
cols = ["Outside_X_Index", "Pixels_Areas", "Sum_of_Luminosity", "X_Perimeter"]
X_train[cols].hist(bins=30, figsize=(12, 7), color="#4C72B0")
plt.suptitle("왜도가 큰 Feature의 분포")
plt.tight_layout()
plt.show()

In [ ]:
# 극단값은 어떤 유형에서 나오는가? (면적 기준, 사분위 범위 1.5배 밖)
q1, q3 = X_train["Pixels_Areas"].quantile([0.25, 0.75])
area_upper = q3 + 1.5 * (q3 - q1)
is_extreme = X_train["Pixels_Areas"] > area_upper
print(f"면적 극단값 기준: {area_upper:.1f} 초과 → {int(is_extreme.sum())}건 (Train의 {is_extreme.mean():.1%})")
(y_train[is_extreme].value_counts(normalize=True) * 100).round(1).to_frame("극단값 중 비율(%)")

**확인 결과**
- `Outside_X_Index`(왜도 5.26), `Sum_of_Luminosity`(3.25), `Pixels_Areas`(3.11), `X_Perimeter`(2.85) 등에서 강한 오른쪽 꼬리가 확인됩니다. 크기 계열 변수의 왜도는 약 2.8 ~ 3.3입니다.
- 면적 기준 극단값(사분위 범위 1.5배 밖) 317건 중 **82.6%가 K_Scratch**입니다.

**판단**
- 큰 값을 측정 오류로 볼 근거가 없고, 극단값 대부분이 K_Scratch라는 것은 **"큰 면적"이 K_Scratch를 알아보는 단서**라는 뜻입니다. 지우면 가장 잘 구분되는 유형의 단서를 지우게 됩니다. → **극단값을 제거하지 않습니다.**
- 주력 모델인 트리 계열은 기준값으로 나누기만 하므로 극단값에 흔들리지 않습니다.

## 6. Feature 간 관계 (README 6.2)

정보가 중복되는 Feature 쌍을 확인합니다. 원본 크기 변수와 로그 변수처럼 **단조 관계**인 경우는 Pearson보다 Spearman이 중복을 더 잘 드러내므로 둘 다 봅니다.

In [ ]:
corr = X_train.corr(method="pearson")
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = upper.stack().reset_index()
pairs.columns = ["Feature_1", "Feature_2", "Pearson"]
pairs = pairs.reindex(pairs["Pearson"].abs().sort_values(ascending=False).index)
pairs.head(15).reset_index(drop=True)

In [ ]:
for a, b in [("Pixels_Areas", "LogOfAreas"), ("Pixels_Areas", "Sum_of_Luminosity"), ("X_Minimum", "X_Maximum"),
             ("Y_Minimum", "Y_Maximum")]:
    print(f"{a:13s} ↔ {b:11s}  Pearson {X_train[a].corr(X_train[b]):.4f}"
          f" | Spearman {X_train[a].corr(X_train[b], method='spearman'):.4f}")

**확인 결과**
- `TypeOfSteel_A300`↔`A400`(−1.000), `Y_Minimum`↔`Y_Maximum`(1.000), `Pixels_Areas`↔`Sum_of_Luminosity`(0.997) 등 위치·크기 계열에서 매우 높은 상관이 있습니다.
- `Pixels_Areas`↔`LogOfAreas`는 Pearson 0.80이지만 Spearman 1.00입니다. 로그 변환으로 선형 관계는 약해졌지만 **값의 순서는 완전히 같습니다.**

**판단**
- 강종 변수 두 개는 **같은 정보를 반대로 표현**하므로 하나를 제거합니다(02).
- 그 외 상관이 높은 Feature는 위치·크기 등 결함의 서로 다른 물리적 특성을 나타내므로 정보 중복이라고 단정하기 어려워 **유지**합니다. 다만 상관이 높은 Feature끼리는 Feature Importance가 나뉘어 보일 수 있다는 점을 해석 시 고려합니다.

## 7. 발견 1. 변수 하나로는 일부 유형만 갈린다 (README 6.1)

**질문:** 결함 유형마다 Feature 분포가 다른가? 하나의 Feature로 모든 유형을 구분할 수 있는가?

Kruskal-Wallis 검정(정규분포 가정이 필요 없는 다집단 비교)으로 유형 간 분포 차이를 확인합니다. 이 데이터는 값이 한쪽으로 몰려 있고 극단값이 많아서, 평균을 비교하는 분산분석보다 **순위**를 비교하는 이 검정이 적합합니다.

H 통계량은 표본 수가 많을수록 커지므로, 크기를 비교할 때는 **효과크기 ε² = (H − k + 1) / (n − k)** (k: 유형 수, n: 표본 수)를 씁니다. 0 ~ 1 사이 값이고 클수록 변수 하나로 유형이 잘 갈린다는 뜻입니다.

In [ ]:
from scipy.stats import kruskal

fault_types = y_train.unique()
k, n = len(fault_types), len(y_train)
kw = []
for col in X_train.columns:
    h, p = kruskal(*[X_train.loc[y_train == c, col] for c in fault_types])
    kw.append({"Feature": col, "Kruskal_H": h, "epsilon2": (h - k + 1) / (n - k), "p_value": p})

# A400은 A300과 같은 정보(효과크기도 같음)이고 02에서 제거하므로 순위에서 뺀다 → 26개 기준
kw = (pd.DataFrame(kw).query("Feature != 'TypeOfSteel_A400'")
      .sort_values("epsilon2", ascending=False).reset_index(drop=True))
kw.index += 1
kw["묶음"] = kw["Feature"].map(feat2group)
print("p < 0.001 인 Feature 수:", int((kw["p_value"] < 0.001).sum()), "/", len(kw))
print("상위 10개 중 크기 변수:", int((kw.head(10)["묶음"] == "크기").sum()), "개")
kw[["Feature", "묶음", "epsilon2", "p_value"]].round(4)

In [ ]:
# 탐색용 그림 (파일로 저장하지 않음. README 그림은 아래 셀에서 저장)
import seaborn as sns

eda = X_train.assign(Fault_Type=y_train.values)
order = y_train.value_counts().index

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, col, title in zip(
        axes,
        ["LogOfAreas", "Orientation_Index", "X_Minimum"],
        ["면적 (LogOfAreas)", "방향 (Orientation_Index)", "가로 위치 (X_Minimum)"]):
    sns.boxplot(data=eda, x="Fault_Type", y=col, order=order, ax=ax,
                color="#9DB4D6", fliersize=2)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=35)
axes[1].axhline(0, ls="--", c="gray", alpha=0.6)

plt.suptitle("결함 유형별 주요 Feature 분포", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
steel = (pd.crosstab(eda["Fault_Type"], eda["TypeOfSteel_A300"], normalize="index") * 100)
steel.columns = ["A400(%)", "A300(%)"]
steel.round(1)

In [ ]:
# README 6.1 발견 1 그림 → figures/eda_result.png
BOX_ORDER = ["K_Scratch", "Z_Scratch", "Stains", "Dirtiness", "Bumps", "Pastry", "Other_Faults"]
OVERLAP = ["Bumps", "Pastry", "Other_Faults"]   # 위 박스플롯에서 여러 Feature 분포가 겹친 세 유형
BLUE, ORANGE = "#6497CD", "#E89C5E"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14.5, 7), gridspec_kw={"width_ratios": [1, 1.1]})

bars = kw.iloc[::-1]                                  # 큰 값이 위로
ax1.barh(bars["Feature"], bars["epsilon2"], color=[GROUP_COLOR[g] for g in bars["묶음"]])
ax1.margins(y=0.01)
ax1.set_xlabel("분리력 (Kruskal–Wallis 효과크기 ε²)")
ax1.set_title("① 변수 하나만으로 결함 유형을 얼마나 가르나", fontweight="bold")
ax1.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in GROUP_COLOR.values()], list(GROUP_COLOR),
           title="변수 묶음", loc="lower right", frameon=False)
ax1.text(0, -0.1, f"모든 변수 p < 0.001 · Train {n:,}건 기준", transform=ax1.transAxes, fontsize=9, color="dimgray")

bp = ax2.boxplot([X_train.loc[y_train == c, "LogOfAreas"] for c in BOX_ORDER],
                 patch_artist=True, widths=0.6, medianprops={"color": "black"},
                 flierprops={"marker": "o", "markersize": 3, "markerfacecolor": "none",
                             "markeredgecolor": "gray", "alpha": 0.7})
for box, c in zip(bp["boxes"], BOX_ORDER):
    box.set_facecolor(ORANGE if c in OVERLAP else BLUE)
ax2.set_xticks(range(1, len(BOX_ORDER) + 1), BOX_ORDER)
ax2.set_ylabel("결함 면적 (로그 값, LogOfAreas)")
ax2.set_title("② 면적만으로는 K형 긁힘·얼룩만 확실히 갈린다", fontweight="bold")

for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_result.png", dpi=150, bbox_inches="tight")
plt.show()

**확인 결과**
- 26개 Feature 모두 유형 간 분포 차이가 통계적으로 유의했습니다(p < 0.001). 다만 표본이 많으면 작은 차이도 유의하게 나오므로, **효과크기와 차이의 모양**을 함께 확인했습니다.
- **효과크기 상위권은 대부분 크기 변수**입니다(상위 10개 중 8개). 결함이 얼마나 큰지가 유형을 가르는 첫 번째 기준입니다.
- 강종(`TypeOfSteel_A300`)과 두께는 상위권이지만, `Length_of_Conveyer`(강판 폭)는 ε² 0.20으로 26개 중 18위입니다. Y 좌표 두 개는 최하위입니다.
- **면적:** K_Scratch는 다른 유형보다 크고 Stains는 매우 작습니다. 나머지 다섯 유형은 상당 부분 겹칩니다.
- **방향:** Pastry·Dirtiness는 양수 쪽, K_Scratch·Stains는 음수 쪽에 몰려 있습니다. 면적에서 겹치던 유형 일부가 방향에서는 구분됩니다.
- **위치:** K_Scratch는 매우 낮은 X 위치(강판 한쪽 가장자리)에 집중됩니다.
- **강종:** K_Scratch·Stains는 100% A400, Z_Scratch는 약 90%가 A300입니다.

**판단**
- Feature마다 구분해주는 유형이 다르므로, **여러 Feature를 함께 조합하는 비선형 모델**(트리 기반)이 유리할 것으로 예상합니다.
- `Length_of_Conveyer`는 혼자서는 분리력이 약하지만 강종·두께와 묶이면 "어느 강판인지"를 알려 주는 정보가 될 수 있습니다. → 8장에서 확인합니다.
- Bumps·Other_Faults·Pastry는 여러 Feature에서 분포가 겹쳐 **서로 혼동될 가능성**이 있습니다. → Baseline 혼동행렬에서 확인합니다.

## 8. 발견 2. 같은 생산 조건에서는 같은 결함이 반복된다 (README 6.1)

강종·두께·폭이 모두 같은 결함들을 하나의 **"생산 조건 묶음"**으로 봅니다. 데이터에 강판 번호가 없어서, 이 세 값이 같으면 같은 강판이거나 같은 규격으로 함께 생산된 강판일 가능성이 높다고 본 것입니다.

| 그림 | 질문 | 사용 데이터 |
|---|---|---|
| ① | 같은 묶음 안에서 같은 유형이 우연 이상으로 몰려 있는가 | Train |
| ② | 강종만으로 유형이 얼마나 정해지는가 | **[전체 데이터]** |
| ③ | 유형마다 나온 `Length_of_Conveyer`의 범위는 어떤가 | **[전체 데이터]** |
| ④ | 처음 보는 생산 조건에서 최종 모델이 더 자주 틀리는가 | **[Test 포함]** 최종 모델의 Test 결과 |

In [ ]:
# 전체 데이터 (②·③·④와 9·10장에서 사용)
all_X = raw.drop(columns=TARGET_COLS)
all_y = raw[TARGET_COLS].idxmax(axis=1).replace({"K_Scatch": "K_Scratch"})
test_idx = pd.read_csv(PROJECT_ROOT / "data" / "split" / "test_idx.csv")["index"]

def group_key(X):
    """생산 조건 묶음: TypeOfSteel_A300 + Steel_Plate_Thickness + Length_of_Conveyer"""
    return (X["TypeOfSteel_A300"].astype(str) + "_" + X["Steel_Plate_Thickness"].astype(str)
            + "_" + X["Length_of_Conveyer"].astype(str))

def purity(groups, labels):
    """묶음마다 가장 많은 유형의 비율을, 묶음 크기로 가중 평균"""
    d = pd.DataFrame({"g": groups.values, "y": labels.values})
    top_share = d.groupby("g")["y"].agg(lambda s: s.value_counts().iloc[0] / len(s))
    size = d.groupby("g").size()
    return (top_share * size).sum() / size.sum()

# ① Train 기준 묶음 구조와 무작위 비교
grp_train = group_key(X_train)
size = grp_train.value_counts()
real_purity = purity(grp_train, y_train)
rng = np.random.default_rng(0)
shuffled = np.array([purity(grp_train, pd.Series(rng.permutation(y_train.values))) for _ in range(200)])

print(f"묶음 수: {len(size)}  묶음 크기 중앙값: {size.median():.0f}  최대: {size.max()}")
print(f"실제 순도: {real_purity:.3f}  무작위 평균/최대: {shuffled.mean():.3f} / {shuffled.max():.3f}")

# Test 결함 중 Train에 같은 묶음이 있는 비율 [Test 포함]
grp_test = group_key(all_X.loc[test_idx])
seen = grp_test.isin(set(grp_train))
print(f"Test 중 Train에 같은 묶음이 있는 비율: {seen.mean():.3f} (처음 보는 묶음 {int((~seen).sum())}건)")

In [ ]:
# ② 유형별 A300 비율, ③ 유형별 Length_of_Conveyer 범위 [전체 데이터]
a300_share = all_X.groupby(all_y)["TypeOfSteel_A300"].mean().sort_values()
width_range = all_X.groupby(all_y)["Length_of_Conveyer"].agg(lambda s: s.max() - s.min()).sort_values()
display(pd.DataFrame({"A300 비율(%)": (a300_share * 100).round(1), "Length_of_Conveyer 범위": width_range}))

# ④ 최종 모델의 Test 오분류 건수 [Test 포함]
# 최종 모델의 예측은 이 노트북에서 만들지 않으므로, process_condition.png 원본 그림의 값을 그대로 씁니다.
# (Train에 있던 묶음 365건 중 67건, 처음 보는 묶음 24건 중 7건. 합계 74건은 README 8.2의 Test 오분류 수와 같음)
FINAL_TEST_ERRORS = {"seen": 67, "unseen": 7}
n_seen, n_unseen = int(seen.sum()), int((~seen).sum())
assert (n_seen, n_unseen) == (365, 24), "분할이 팀 공통 분할과 다릅니다."
err_rate = {"seen": FINAL_TEST_ERRORS["seen"] / n_seen, "unseen": FINAL_TEST_ERRORS["unseen"] / n_unseen}
print(f"오분류율: Train에 있던 조건 {err_rate['seen']:.1%} / 처음 보는 조건 {err_rate['unseen']:.1%}")

In [ ]:
# README 6.1 발견 2 그림 → figures/process_condition.png
CONFUSED = ["Bumps", "Pastry", "Other_Faults"]
RED, GRAY, BLUE2, ORANGE2 = "#C9443C", "#AEB4BC", "#4F86C6", "#E8914A"
fig, axes = plt.subplots(2, 2, figsize=(14, 9.6))

# ①
ax = axes[0, 0]
ax.hist(shuffled, bins=30, color=GRAY, label="정답을 무작위로 섞었을 때 (200회)")
ax.axvline(real_purity, color=RED, lw=3, label="실제 데이터")
ax.text(shuffled.mean() + 0.01, ax.get_ylim()[1] * 0.9, f"무작위 평균 {shuffled.mean():.3f}", color="dimgray", fontsize=11)
ax.text(real_purity - 0.005, ax.get_ylim()[1] * 0.9, f"실제 {real_purity:.3f}", color=RED, fontsize=11,
        fontweight="bold", ha="right")
ax.set_xlim(0.39, 0.81)
ax.set_xlabel("같은 공정 조건 묶음 안에서 가장 많은 유형이 차지하는 비율")
ax.set_ylabel("횟수")
ax.legend(loc="center right", frameon=False, fontsize=9)
ax.set_title("① 같은 공정 조건이면 같은 결함 유형이 반복된다", fontweight="bold")

# ②
ax = axes[0, 1]
order = a300_share.index[::-1]                       # A300 비율이 낮은 유형이 위로
ax.barh(order, 1 - a300_share[order], color="#5B8DB8", label="A400 강종")
ax.barh(order, a300_share[order], left=1 - a300_share[order], color="#D98B5F", label="A300 강종")
for i, c in enumerate(order):
    ax.text(1.01, i, f"A300 {a300_share[c]:.0%}", va="center", fontsize=9)
ax.set_xlim(0, 1.18)
ax.set_xticks([0, 0.5, 1], ["0%", "50%", "100%"])
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2, frameon=False, fontsize=9)
ax.set_title("② 강종만 알아도 일부 유형은 거의 정해진다", fontweight="bold")

# ③
ax = axes[1, 0]
order = width_range.index[::-1]                      # 범위가 좁은 유형이 위로
jitter = np.random.default_rng(0)
for i, c in enumerate(order):
    w = all_X.loc[all_y == c, "Length_of_Conveyer"]
    ax.plot([w.min(), w.max()], [i, i], lw=7, color=ORANGE2 if c in CONFUSED else BLUE2, solid_capstyle="butt")
    ax.scatter(w, i + jitter.uniform(-0.18, 0.18, len(w)), s=6, color="black", alpha=0.3, zorder=3)
    ax.text(1915, i, f"범위 {width_range[c]}", va="center", fontsize=9)
ax.set_yticks(range(len(order)), order)
ax.set_xlim(1200, 1900)
ax.set_xlabel("강판 폭 (원래 열 이름 Length_of_Conveyer)")
ax.set_title("③ 얼룩(Stains)은 사실상 한 가지 폭에서만 나왔다", fontweight="bold")

# ④
ax = axes[1, 1]
vals = [err_rate["seen"], err_rate["unseen"]]
ax.bar([0, 1], vals, width=0.55, color=["#2E6FB7", RED])
for x, v, cnt in zip([0, 1], vals, [n_seen, n_unseen]):
    ax.text(x, v + 0.006, f"오분류율 {v:.1%}\n(Test {cnt}건)", ha="center", va="bottom", fontsize=10)
ax.set_xticks([0, 1], ["Train에 있던\n공정 조건", "Train에 없던\n(처음 보는) 공정 조건"])
ax.set_ylim(0, 0.4)
ax.set_ylabel("최종 모델 Test 오분류율")
ax.set_title("④ 처음 보는 공정 조건에서는 더 자주 틀린다", fontweight="bold")

for ax in axes.flat:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(h_pad=3)
plt.savefig(FIG_DIR / "process_condition.png", dpi=150, bbox_inches="tight")
plt.show()

**확인 결과**
- Train에는 생산 조건 묶음이 218개 있고, 묶음 하나당 결함 수는 중앙값 3건, 최대 108건입니다.
- 묶음 안에서 가장 많은 유형이 차지하는 비율은 **0.763**으로, 정답을 무작위로 섞었을 때(200회 평균 0.431, 최대 0.453)보다 훨씬 높습니다.
- 강종만 알아도 K_Scratch·Stains(거의 A400)와 Z_Scratch(대부분 A300)는 거의 정해지고, Stains는 `Length_of_Conveyer` 범위가 8로 사실상 한 가지 폭에서만 나왔습니다.
- Test 결함의 93.8%는 Train에 같은 묶음이 있습니다. 처음 보는 묶음에서 나온 24건은 최종 모델의 오분류율이 더 높았지만, 건수가 적어 이것만으로 결론 내리지 않습니다(그룹 교차검증은 05 노트북, README 8.4).

**판단**
- 모델은 결함의 생김새를 배우지 않아도 "이 강판에서는 보통 이 결함이 나온다"는 지름길로 점수를 낼 수 있습니다. → 일반 교차검증 점수는 "이미 본 생산 조건" 기준으로 읽고, 처음 보는 조건은 그룹 교차검증으로 따로 확인합니다.

## 9. 발견 3. 모양만으로는 헷갈리는 유형이 많다 (README 6.1)

위치·공정 정보를 빼고 **모양·밝기 특징 11개만으로** 결함끼리 얼마나 닮았는지 계산합니다. 각 결함마다 가장 닮은 이웃 5개를 찾아, 그중 같은 유형이 얼마나 되는지 봅니다. 이어서 공정 조건 3개(강종·두께·폭)를 더해 같은 계산을 반복합니다.

- 사용 데이터: **[전체 데이터]** (README 그림과 같은 기준). 표준화 후 유클리드 거리로 이웃을 찾습니다.
- 그림의 빨간 마름모는 참고용 최종 모델 Test F1이며 **[Test 포함]**, 이 분석의 판단에는 쓰지 않습니다.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

SHAPE = ["LogOfAreas", "Log_X_Index", "Log_Y_Index", "Orientation_Index", "Square_Index", "Empty_Index",
         "Edges_X_Index", "Edges_Y_Index", "Luminosity_Index", "Minimum_of_Luminosity", "Maximum_of_Luminosity"]
PROC = ["TypeOfSteel_A300", "Steel_Plate_Thickness", "Length_of_Conveyer"]
labels = all_y.to_numpy()

def neighbor_ratio(cols, k=5):
    """각 결함의 가장 닮은 이웃 k개 중 같은 유형의 비율 (유형별 평균)"""
    Z = StandardScaler().fit_transform(all_X[cols])
    _, ind = NearestNeighbors(n_neighbors=k + 1).fit(Z).kneighbors(Z)
    same = (labels[ind[:, 1:]] == labels[:, None]).mean(axis=1)   # 자기 자신 제외
    return pd.Series(same).groupby(labels).mean()

# 최종 모델의 유형별 Test F1 (Taegyeom_08_train_final_model.ipynb 출력, README 8.2). 그림의 참고 표시에만 사용.
FINAL_TEST_F1 = {"Other_Faults": 0.7714, "Bumps": 0.7421, "K_Scratch": 0.9536, "Z_Scratch": 0.9730,
                 "Pastry": 0.5455, "Stains": 0.8889, "Dirtiness": 0.9524}

similarity = pd.DataFrame({"모양·밝기만": neighbor_ratio(SHAPE), "+ 공정 조건": neighbor_ratio(SHAPE + PROC)})
similarity["증가"] = similarity["+ 공정 조건"] - similarity["모양·밝기만"]
similarity = similarity.sort_values("모양·밝기만", ascending=False)
similarity.round(3)

In [ ]:
# README 6.1 발견 3 그림 → figures/shape_similarity.png
fig, ax = plt.subplots(figsize=(14, 5.6))
x = np.arange(len(similarity))
w = 0.36
b1 = ax.bar(x - w / 2, similarity["모양·밝기만"], w, color="#9DB8D9", label="모양·밝기 특징 11개만 사용")
b2 = ax.bar(x + w / 2, similarity["+ 공정 조건"], w, color="#2E6FB7", label="+ 공정 조건 3개 (강종·두께·폭)")
for bars in (b1, b2):
    ax.bar_label(bars, labels=[f"{v:.0%}" for v in bars.datavalues], padding=3, fontsize=9)
sc = ax.scatter(x, [FINAL_TEST_F1[c] for c in similarity.index], marker="D", s=110, color="#C9443C",
           zorder=3, label="참고: 최종 모델 Test F1")

top = similarity["증가"].idxmax()                    # 공정 조건을 더했을 때 가장 크게 뛰는 유형
i = list(similarity.index).index(top)
ax.annotate("공정 조건을 더하면\n가장 크게 뛰는 유형", xy=(i + w / 2, similarity.loc[top, "+ 공정 조건"] + 0.01),
            xytext=(i - 1.3, 1.0), ha="center", fontsize=10, arrowprops={"arrowstyle": "->", "color": "#333333"})

ax.set_xticks(x, similarity.index)
ax.set_ylim(0, 1.08)
ax.set_ylabel("가장 닮은 결함 5개 중 같은 유형의 비율")
ax.set_title("모양만 보면 헷갈리는 유형이 많다 — 특히 Z형 긁힘은 공정 조건을 더해야 구분된다", fontweight="bold")
ax.legend([sc, b1, b2], [sc.get_label(), b1.get_label(), b2.get_label()],
          loc="upper center", frameon=False, fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / "shape_similarity.png", dpi=150, bbox_inches="tight")
plt.show()

**확인 결과**
- K_Scratch(91%)와 Stains(82%)는 모양만으로도 이웃 대부분이 같은 유형입니다.
- Bumps·Pastry·Other_Faults는 모양이 닮은 이웃의 절반 이상이 다른 유형입니다.
- Z_Scratch는 모양만으로는 가장 헷갈리는 유형(32%)인데, 공정 조건을 더하면 69%로 가장 크게 뜁니다. Dirtiness도 44% → 67%로 크게 오릅니다.

**판단 (가설)**
- 최종 모델이 Z_Scratch를 가장 잘 맞히는 것은 생김새보다 **"주로 A300 강종에서 나오는 결함"**이라는 공정 조건 정보를 활용한 결과일 가능성이 있습니다. 모델이 실제로 공정 조건에 얼마나 기대는지는 Feature Importance(README 8.3)와 그룹 교차검증(README 8.4)으로 확인합니다.

## 10. 발견 4. K_Scratch 일부는 같은 자리에서 일정한 간격으로 반복된다 (README 6.1)

같은 생산 조건 묶음 안에서 같은 가로 위치에 줄지어 있는 결함을 찾고, 길이 방향(Y) 간격에 규칙이 있는지 확인합니다.

- 대상: `Length_of_Conveyer` 1358 · 두께 40 · A400 묶음에서 결함 가로 중심이 125 ± 20인 결함 **[전체 데이터]**. Train·Test 위치는 그림에서 구분해 표시합니다 **[Test 포함]**.
- 방법: 이웃한 결함 사이 간격이 어떤 기본 주기의 정수배(오차 ±15)에 맞는 비율을, 주기를 1,000 ~ 20,000 범위에서 찾아 가장 높은 값으로 봅니다.
- 비교: 같은 개수의 점을 무작위로 뿌린 뒤 ① 같은 주기를 적용했을 때, ② 주기까지 다시 찾아 줬을 때(무작위에 유리한 조건)의 비율.
- 실행 시간: 약 2분

In [ ]:
xc = (all_X["X_Minimum"] + all_X["X_Maximum"]) / 2
col_mask = ((all_X["Length_of_Conveyer"] == 1358) & (all_X["Steel_Plate_Thickness"] == 40)
            & (all_X["TypeOfSteel_A300"] == 0) & ((xc - 125).abs() <= 20))
col = all_X[col_mask]
y_pos = col["Y_Minimum"].values

def fit_rate(yy, p, tol=15):
    """이웃한 결함 사이 간격 중, 주기 p의 정수배(오차 ±tol)에 맞는 비율"""
    d = np.diff(np.sort(yy)); d = d[d > 0]
    r = d / p
    return np.mean(np.abs(r - np.round(r)) * p <= tol)

best_rate, best_p = max((fit_rate(y_pos, p), p) for p in np.arange(1000, 20000, 0.5))

rng = np.random.default_rng(0)
rand_fixed = np.mean([fit_rate(rng.uniform(y_pos.min(), y_pos.max(), len(y_pos)), best_p) for _ in range(200)])
rand_search = np.mean([max(fit_rate(yy, p) for p in np.arange(1000, 20000, 2))
                       for yy in [rng.uniform(y_pos.min(), y_pos.max(), len(y_pos)) for _ in range(20)]])

in_test = col.index.isin(test_idx)
print(f"결함 {len(col)}건 {all_y[col.index].value_counts().to_dict()}  (Train {int((~in_test).sum())} / Test {int(in_test.sum())})")
print(f"최적 주기: {best_p}  적합률: {best_rate:.3f}")
print(f"무작위(같은 주기): {rand_fixed:.3f}  무작위(주기 다시 탐색): {rand_search:.3f}")

In [ ]:
# README 6.1 발견 4 그림 → figures/roll_mark_period.png
fig = plt.figure(figsize=(14, 8.8))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.15], hspace=0.35)
ax1 = fig.add_subplot(gs[0, :]); ax2 = fig.add_subplot(gs[1, 0]); ax3 = fig.add_subplot(gs[1, 1])

# ① 위치
ax1.scatter(col.loc[~in_test, "Y_Minimum"] / 1e6, xc[col.index][~in_test], s=30, color="#2E6FB7", label="Train에 들어간 결함")
ax1.scatter(col.loc[in_test, "Y_Minimum"] / 1e6, xc[col.index][in_test], s=45, marker="^", color="#C9443C", label="Test에 들어간 결함")
ax1.set_ylim(80, 170)
ax1.set_xlabel("강판 길이 방향 위치 (Y_Minimum, 백만 단위)")
ax1.set_ylabel("가로 위치 (X 중심)")
ax1.legend(loc="upper right", ncol=2, frameon=False)
ax1.set_title(f"① 같은 강판 묶음(폭 1358 · 두께 40 · A400)의 왼쪽 가장자리에 K형 긁힘 {len(col)}건이 한 줄로 늘어서 있다",
              fontweight="bold")

# ② 간격 ÷ 주기
gaps = np.diff(np.sort(y_pos)); gaps = gaps[gaps > 0] / best_p
for k in range(1, 31):
    ax2.axvline(k, color="gray", ls=":", lw=0.8)
ax2.hist(gaps, bins=np.arange(0, 30.25, 0.25), color="#2E6FB7")
ax2.set_xlim(0, 30)
ax2.set_xlabel(f"이웃한 결함 사이 간격 ÷ 기본 주기({best_p:,.0f})")
ax2.set_ylabel("간격 수")
ax2.set_title("② 간격이 기본 주기의 정수배 자리(점선)에만 몰려 있다", fontweight="bold")

# ③ 무작위 비교
vals = [best_rate, rand_search, rand_fixed]
bars = ax3.bar([0, 1, 2], vals, width=0.55, color=["#C9443C", "#9AA3AD", "#C8CDD3"])
ax3.bar_label(bars, labels=[f"{v:.0%}" for v in vals], padding=3, fontweight="bold")
ax3.set_xticks([0, 1, 2], ["실제 데이터", "무작위 배치\n(주기도 다시 찾아줌)", "무작위 배치\n(같은 주기 적용)"])
ax3.set_ylim(0, 1)
ax3.set_ylabel("간격 중 주기의 정수배와 맞는 비율 (오차 ±15)")
ax3.set_title("③ 우연으로는 나오기 어려운 규칙성", fontweight="bold")

for ax in (ax1, ax2, ax3):
    ax.spines[["top", "right"]].set_visible(False)
plt.savefig(FIG_DIR / "roll_mark_period.png", dpi=150, bbox_inches="tight")
plt.show()

**확인 결과**
- 한 강판 묶음의 왼쪽 가장자리를 따라 K_Scratch 92건이 한 줄로 늘어서 있습니다. 이 중 19건은 Test, 73건은 Train에 들어갔습니다.
- 이웃한 결함 사이 간격의 81%가 하나의 기본 주기의 정수배에 맞습니다. 같은 개수의 점을 무작위로 뿌리면 주기를 다시 찾아 줘도 8%, 같은 주기를 적용하면 1% 수준입니다.

**해석 (가설)**
- 롤 표면의 한 지점에 흠이 있으면 롤이 한 바퀴 돌 때마다 같은 가로 위치에 같은 흠이 찍히는 **롤 마크**의 전형적인 패턴과 일치합니다. 유형 정의가 공개되지 않아 롤 마크라고 확정할 수는 없습니다.
- 이런 "쌍둥이 결함"이 무작위 분할로 Train과 Test에 나뉘어 들어가므로, 일반 교차검증과 Test 점수는 실제보다 좋게 나올 수 있습니다.
- 주기가 이렇게 정확히 맞는다는 것은 Y가 강판을 따라 잰 실제 거리라는 근거입니다.

다른 묶음·위치에 대한 같은 분석(README 6.1 발견 4 표의 나머지 행)은 `dongjin_data_quality.ipynb`에 있습니다.

## 11. Other_Faults 내부 특성 (README 8.2)

Other_Faults는 가장 많은 유형이면서 "기타"라는 성격상 서로 다른 결함이 섞여 있을 수 있습니다.
**질문:** Other_Faults는 다른 유형보다 Feature 값이 넓게 퍼져 있는가?

각 유형의 Feature별 IQR(사분위 범위)을 전체 Train IQR로 나눈 **상대 IQR**로 비교합니다. 1에 가까울수록 그 유형 하나가 전체만큼 넓게 퍼져 있다는 뜻입니다.

In [ ]:
total_iqr = X_train.quantile(0.75) - X_train.quantile(0.25)
rel_iqr = pd.DataFrame({
    c: (X_train[y_train == c].quantile(0.75) - X_train[y_train == c].quantile(0.25)) / total_iqr
    for c in y_train.unique()
}).replace([np.inf, -np.inf], np.nan)

rel_iqr.median().sort_values(ascending=False).to_frame("상대 IQR 중앙값")

In [ ]:
compare = pd.DataFrame({
    "Other_Faults": rel_iqr["Other_Faults"],
    "나머지 유형 중앙값": rel_iqr.drop(columns="Other_Faults").median(axis=1),
})
compare["배수"] = compare["Other_Faults"] / compare["나머지 유형 중앙값"]
compare.replace([np.inf], np.nan).dropna().sort_values("배수", ascending=False).head(8)

**확인 결과**
- Other_Faults의 상대 IQR 중앙값은 0.98로 7개 유형 중 가장 큽니다.
- 특히 `Length_of_Conveyer`, `Steel_Plate_Thickness` 등 공정 관련 Feature에서 다른 유형보다 훨씬 넓게 퍼져 있습니다. 반면 `Pixels_Areas` 등 크기 Feature에서는 그렇지 않습니다.

**해석 (가설)**
- Other_Faults는 여러 공정 조건에서 발생한 다양한 결함을 포함할 **가능성**이 있습니다.
- 다만 이 결과만으로 내부에 별도 하위 유형이 있다고 판단할 수는 없으므로 클래스를 나누지는 않습니다.

## 12. EDA 결론 및 전처리·모델링 결정

| 구분 | 내용 |
|---|---|
| **확인된 사실** | 결측·완전 중복 0건 / 원-핫 합 = 1 / 강종 2개 열 완전 중복 / `Length_of_Conveyer`는 강판(검사 영역)의 가로 폭 / 비율·모양 지표 13개는 원래 측정값으로 계산된 값 / `Sum_of_Luminosity`는 사실상 크기 변수 / 크기 Feature 강한 우측 왜도, 면적 극단값의 82.6%가 K_Scratch / 위치·크기 계열 높은 상관 / 효과크기 상위권은 대부분 크기 변수 / 같은 생산 조건 묶음 안의 유형 쏠림(0.763 vs 무작위 0.431) / 최대·최소 유형 12.23배 차이 (Train 기준) |
| **해석** | 극단값은 K_Scratch의 특징 / 단일 Feature로는 일부 유형만 구분됨 / Bumps·Other_Faults·Pastry는 분포가 겹침 / 공정 조건이 "어느 강판인지"를 알려 주는 지름길이 될 수 있음 |
| **가설 (추가 검증 필요)** | K_Scratch 일부는 롤 마크 / Z_Scratch는 공정 조건 덕분에 구분됨 / Other_Faults 내부 이질성 |

**02 전처리로 넘기는 결정**
1. 원-핫 7개 열 → 단일 `target` 생성 후 원-핫 열 제거 (누수 방지)
2. `K_Scatch` → `K_Scratch` 표기 수정
3. `TypeOfSteel_A400` 제거 (`A300` 하나로 동일 정보 표현)
4. 결측·완전 중복·극단값 처리 없음 (극단값은 K_Scratch의 특징)
5. 유사 중복은 Train·Test에 걸쳐 있을 수 있으므로 02에서 전체 데이터 기준으로 확인

**03 모델링으로 넘기는 결정**
- 핵심 지표: **Macro F1** (+ 유형별 Precision·Recall·F1, 혼동행렬)
- Scaling은 Logistic Regression에만, Pipeline 안에서 적용
- Bumps·Pastry·Other_Faults 간 혼동을 Baseline에서 중점 확인
- 같은 생산 조건 묶음이 학습·검증에 섞이지 않는 **그룹 교차검증**으로 처음 보는 조건의 성능을 따로 확인